<a href="https://colab.research.google.com/github/dawn-pixellate/activepieces/blob/main/n1npu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install amaranth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.3/254.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 5.0 MB/s eta 0:00:00


In [3]:
from amaranth import *
from amaranth.sim import Simulator

class EdgeAdder(Elaboratable):
    def __init__(self):
        # Define the physical input pins (8-bit width)
        self.a = Signal(8)
        self.b = Signal(8)
        # Define the output pin (9-bit width to prevent hardware overflow)
        self.out = Signal(9)

    def elaborate(self, platform):
        m = Module()
        # Combinatorial logic: physically wire 'out' to equal 'a + b'
        m.d.comb += self.out.eq(self.a + self.b)
        return m

# 1. Instantiate the physical silicon module
dut = EdgeAdder()

# 2. Spin up the hardware simulator
sim = Simulator(dut)

# 3. Write a testbench to push electrical states through the pins
def process():
    print("⚡ Powering up simulated silicon...")

    # Test Cycle 1
    yield dut.a.eq(42)
    yield dut.b.eq(15)
    yield # Advance the hardware clock tick
    out_val = yield dut.out
    print(f"Cycle 1 [42 + 15]: Gate output is {out_val}")

    # Test Cycle 2 (Testing near 8-bit limits)
    yield dut.a.eq(200)
    yield dut.b.eq(50)
    yield # Advance the hardware clock tick
    out_val = yield dut.out
    print(f"Cycle 2 [200 + 50]: Gate output is {out_val}")

sim.add_process(process)

# 4. Run the simulation and write the physical waveform to a file
with sim.write_vcd("lacesse_npu_test.vcd"):
    sim.run()

print("\n✅ Hardware simulation complete. Waveform saved as 'lacesse_npu_test.vcd'")

⚡ Powering up simulated silicon...


TypeError: Received default command from process '/tmp/ipykernel_3846/755281264.py:31' that was added with add_process(); did you mean to use Tick() instead?

In [4]:
from amaranth import *
from amaranth.sim import Simulator, Delay

class EdgeAdder(Elaboratable):
    def __init__(self):
        # Define the physical input pins (8-bit width)
        self.a = Signal(8)
        self.b = Signal(8)
        # Define the output pin (9-bit width to prevent hardware overflow)
        self.out = Signal(9)

    def elaborate(self, platform):
        m = Module()
        # Combinatorial logic: physically wire 'out' to equal 'a + b'
        m.d.comb += self.out.eq(self.a + self.b)
        return m

# 1. Instantiate the physical silicon module
dut = EdgeAdder()

# 2. Spin up the hardware simulator
sim = Simulator(dut)

# 3. Write a testbench to push electrical states through the pins
def process():
    print("⚡ Powering up simulated silicon...")

    # Test Cycle 1
    yield dut.a.eq(42)
    yield dut.b.eq(15)
    # Explicitly wait for the electrical signals to physically propagate through the gates
    yield Delay(1e-6)
    out_val = yield dut.out
    print(f"Cycle 1 [42 + 15]: Gate output is {out_val}")

    # Test Cycle 2 (Testing near 8-bit limits)
    yield dut.a.eq(200)
    yield dut.b.eq(50)
    yield Delay(1e-6)
    out_val = yield dut.out
    print(f"Cycle 2 [200 + 50]: Gate output is {out_val}")

sim.add_process(process)

# 4. Run the simulation and write the physical waveform to a file
with sim.write_vcd("lacesse_npu_test.vcd"):
    sim.run()

print("\n✅ Hardware simulation complete. Waveform saved as 'lacesse_npu_test.vcd'")

⚡ Powering up simulated silicon...
Cycle 1 [42 + 15]: Gate output is 57
Cycle 2 [200 + 50]: Gate output is 250

✅ Hardware simulation complete. Waveform saved as 'lacesse_npu_test.vcd'


In [5]:
from amaranth import *
from amaranth.sim import Simulator

class TernaryMAC(Elaboratable):
    def __init__(self):
        # 1. Physical Pins
        self.x = Signal(signed(8))     # 8-bit Input Activation
        self.w = Signal(signed(2))     # 2-bit Ternary Weight (-1, 0, 1)
        self.clear = Signal()          # Pin to reset the sum for a new calculation
        self.en = Signal()             # Pin to enable the core

        # 2. On-chip Register (Accumulator)
        self.acc = Signal(signed(16))  # 16-bit width prevents data overflow during summing

    def elaborate(self, platform):
        m = Module()

        # Synchronous logic: This physically generates a Flip-Flop register tied to the clock tree
        with m.If(self.clear):
            m.d.sync += self.acc.eq(0)
        with m.Elif(self.en):
            with m.If(self.w == 1):
                m.d.sync += self.acc.eq(self.acc + self.x)
            with m.Elif(self.w == -1):
                m.d.sync += self.acc.eq(self.acc - self.x)
            # Notice there is no logic for w == 0.
            # In hardware, doing nothing saves power! The register simply holds its state.

        return m

# Instantiate and build the simulator
dut = TernaryMAC()
sim = Simulator(dut)

# We are building a stateful chip, so we MUST wire up a physical clock pulse (1 Megahertz)
sim.add_clock(1e-6)

def process():
    print("⚡ Booting Ternary MAC Unit...")

    # 1. Clear the register
    yield dut.clear.eq(1)
    yield dut.en.eq(0)
    yield Tick() # Advance clock 1 tick

    # 2. Start calculation (Dot Product Sequence)
    yield dut.clear.eq(0)
    yield dut.en.eq(1)

    # Cycle 1: w=1, x=50  --> acc = 0 + 50 = 50
    yield dut.w.eq(1)
    yield dut.x.eq(50)
    yield Tick()

    # Cycle 2: w=-1, x=20 --> acc = 50 - 20 = 30
    yield dut.w.eq(-1)
    yield dut.x.eq(20)
    yield Tick()

    # Cycle 3: w=0, x=100 --> acc = 30 (Does nothing, ignores the 100)
    yield dut.w.eq(0)
    yield dut.x.eq(100)
    yield Tick()

    # Cycle 4: w=1, x=15  --> acc = 30 + 15 = 45
    yield dut.w.eq(1)
    yield dut.x.eq(15)
    yield Tick()

    # Read the final physical register state
    final_val = yield dut.acc
    print(f"🏁 Final Accumulator State: {final_val}")

sim.add_sync_process(process)

with sim.write_vcd("lacesse_ternary_mac.vcd"):
    sim.run()

print("✅ Ternary MAC simulation complete. Waveform saved.")

⚡ Booting Ternary MAC Unit...


/tmp/ipykernel_3846/3265860650.py:74: DeprecationWarning: The `add_sync_process` method is deprecated per RFC 27. Use `add_process` or `add_testbench` instead.
  sim.add_sync_process(process)


NameError: name 'Tick' is not defined

In [6]:
from amaranth import *
from amaranth.sim import Simulator, Tick

class TernaryMAC(Elaboratable):
    def __init__(self):
        # 1. Physical Pins
        self.x = Signal(signed(8))     # 8-bit Input Activation
        self.w = Signal(signed(2))     # 2-bit Ternary Weight (-1, 0, 1)
        self.clear = Signal()          # Pin to reset the sum for a new calculation
        self.en = Signal()             # Pin to enable the core

        # 2. On-chip Register (Accumulator)
        self.acc = Signal(signed(16))  # 16-bit width prevents data overflow

    def elaborate(self, platform):
        m = Module()

        # Synchronous logic: This physically generates a Flip-Flop register tied to the clock tree
        with m.If(self.clear):
            m.d.sync += self.acc.eq(0)
        with m.Elif(self.en):
            with m.If(self.w == 1):
                m.d.sync += self.acc.eq(self.acc + self.x)
            with m.Elif(self.w == -1):
                m.d.sync += self.acc.eq(self.acc - self.x)
            # No logic for w == 0. The register simply holds its state, saving power.

        return m

# Instantiate and build the simulator
dut = TernaryMAC()
sim = Simulator(dut)

# Wire up a physical clock pulse (1 Megahertz)
sim.add_clock(1e-6)

def process():
    print("⚡ Booting Ternary MAC Unit...")

    # 1. Clear the register
    yield dut.clear.eq(1)
    yield dut.en.eq(0)
    yield Tick() # Advance clock 1 tick

    # 2. Start calculation (Dot Product Sequence)
    yield dut.clear.eq(0)
    yield dut.en.eq(1)

    # Cycle 1: w=1, x=50  --> acc = 0 + 50 = 50
    yield dut.w.eq(1)
    yield dut.x.eq(50)
    yield Tick()

    # Cycle 2: w=-1, x=20 --> acc = 50 - 20 = 30
    yield dut.w.eq(-1)
    yield dut.x.eq(20)
    yield Tick()

    # Cycle 3: w=0, x=100 --> acc = 30 (Does nothing, ignores the 100)
    yield dut.w.eq(0)
    yield dut.x.eq(100)
    yield Tick()

    # Cycle 4: w=1, x=15  --> acc = 30 + 15 = 45
    yield dut.w.eq(1)
    yield dut.x.eq(15)
    yield Tick()

    # Read the final physical register state
    final_val = yield dut.acc
    print(f"🏁 Final Accumulator State: {final_val}")

# Using the modern RFC 27 API
sim.add_testbench(process)

with sim.write_vcd("lacesse_ternary_mac.vcd"):
    sim.run()

print("✅ Ternary MAC simulation complete. Waveform saved.")

⚡ Booting Ternary MAC Unit...
🏁 Final Accumulator State: 45
✅ Ternary MAC simulation complete. Waveform saved.


In [7]:
from amaranth import *
from amaranth.sim import Simulator, Tick

# 1. The Modified MAC (Now with Data Forwarding)
class SystolicMAC(Elaboratable):
    def __init__(self):
        self.x_in = Signal(signed(8))
        self.x_out = Signal(signed(8)) # Passes X to the next MAC
        self.w = Signal(signed(2))
        self.clear = Signal()
        self.en = Signal()
        self.acc = Signal(signed(16))

    def elaborate(self, platform):
        m = Module()

        # THE SYSTOLIC PULSE: Pass x_in to x_out synchronously on the clock tick
        m.d.sync += self.x_out.eq(self.x_in)

        # The accumulation logic
        with m.If(self.clear):
            m.d.sync += self.acc.eq(0)
        with m.Elif(self.en):
            with m.If(self.w == 1):
                m.d.sync += self.acc.eq(self.acc + self.x_in)
            with m.Elif(self.w == -1):
                m.d.sync += self.acc.eq(self.acc - self.x_in)

        return m

# 2. The 2x2 Array Network
class SystolicArray2x2(Elaboratable):
    def __init__(self):
        # Array inputs (2 rows)
        self.row0_in = Signal(signed(8))
        self.row1_in = Signal(signed(8))

        # Pre-loaded weights
        self.w00 = Signal(signed(2))
        self.w01 = Signal(signed(2))
        self.w10 = Signal(signed(2))
        self.w11 = Signal(signed(2))

        self.clear = Signal()
        self.en = Signal()

        # Readout registers
        self.out00 = Signal(signed(16))
        self.out01 = Signal(signed(16))
        self.out10 = Signal(signed(16))
        self.out11 = Signal(signed(16))

    def elaborate(self, platform):
        m = Module()

        # Instantiate 4 MAC units
        m.submodules.mac00 = mac00 = SystolicMAC()
        m.submodules.mac01 = mac01 = SystolicMAC()
        m.submodules.mac10 = mac10 = SystolicMAC()
        m.submodules.mac11 = mac11 = SystolicMAC()

        # Wire up control signals and weights
        m.d.comb += [
            mac00.clear.eq(self.clear), mac00.en.eq(self.en), mac00.w.eq(self.w00),
            mac01.clear.eq(self.clear), mac01.en.eq(self.en), mac01.w.eq(self.w01),
            mac10.clear.eq(self.clear), mac10.en.eq(self.en), mac10.w.eq(self.w10),
            mac11.clear.eq(self.clear), mac11.en.eq(self.en), mac11.w.eq(self.w11),
        ]

        # WIRING THE DATAPATH (This is the physical array structure)
        m.d.comb += [
            # Row 0 Data Flow
            mac00.x_in.eq(self.row0_in),
            mac01.x_in.eq(mac00.x_out), # Data flows from MAC00 -> MAC01

            # Row 1 Data Flow
            mac10.x_in.eq(self.row1_in),
            mac11.x_in.eq(mac10.x_out)  # Data flows from MAC10 -> MAC11
        ]

        # Map accumulators to array outputs
        m.d.comb += [
            self.out00.eq(mac00.acc), self.out01.eq(mac01.acc),
            self.out10.eq(mac10.acc), self.out11.eq(mac11.acc)
        ]

        return m

# 3. The Testbench
dut = SystolicArray2x2()
sim = Simulator(dut)
sim.add_clock(1e-6)

def process():
    yield dut.clear.eq(1)
    yield dut.en.eq(0)
    yield Tick()
    yield dut.clear.eq(0)
    yield dut.en.eq(1)

    # Load Weights (Row 0: [1, -1], Row 1: [0, 1])
    yield dut.w00.eq(1)
    yield dut.w01.eq(-1)
    yield dut.w10.eq(0)
    yield dut.w11.eq(1)

    print("⚡ Pumping Data through the Array...")

    # Tick 1: Push first column of data
    yield dut.row0_in.eq(10)
    yield dut.row1_in.eq(20)
    yield Tick()

    # Tick 2: Push second column. The first column naturally flows to the right.
    yield dut.row0_in.eq(5)
    yield dut.row1_in.eq(15)
    yield Tick()

    # Tick 3: Push zeros to flush the remaining data through the right side
    yield dut.row0_in.eq(0)
    yield dut.row1_in.eq(0)
    yield Tick()

    # Read final results
    print(f"Row 0 Accumulators: MAC00={yield dut.out00}, MAC01={yield dut.out01}")
    print(f"Row 1 Accumulators: MAC10={yield dut.out10}, MAC11={yield dut.out11}")

sim.add_testbench(process)

with sim.write_vcd("lacesse_systolic_array.vcd"):
    sim.run()

⚡ Pumping Data through the Array...
Row 0 Accumulators: MAC00=15, MAC01=-15
Row 1 Accumulators: MAC10=0, MAC11=35


In [8]:
from amaranth import *

# Assuming SystolicArray2x2 from our previous step is imported

class FikraCFU(Elaboratable):
    def __init__(self):
        # 1. Standard RISC-V CFU Interface Pins
        self.cmd_valid = Signal()
        self.cmd_func_id = Signal(10)
        self.cmd_rs1 = Signal(32) # Input register 1 from CPU
        self.cmd_rs2 = Signal(32) # Input register 2 from CPU

        self.rsp_valid = Signal()
        self.rsp_rd = Signal(32)  # Output register back to CPU

    def elaborate(self, platform):
        m = Module()

        # Instantiate our custom array
        m.submodules.array = array = SystolicArray2x2()

        # Default state: response is not valid until we compute
        m.d.sync += self.rsp_valid.eq(0)

        # When the CPU sends a command...
        with m.If(self.cmd_valid):
            # Function ID 1: Load Weights into the array
            with m.If(self.cmd_func_id == 1):
                # We unpack the 32-bit rs1 into four 8-bit weights for our 2x2 array
                m.d.comb += [
                    array.w00.eq(self.cmd_rs1[0:8]),
                    array.w01.eq(self.cmd_rs1[8:16]),
                    array.w10.eq(self.cmd_rs1[16:24]),
                    array.w11.eq(self.cmd_rs1[24:32]),
                ]
                m.d.sync += self.rsp_valid.eq(1) # Tell CPU we are done
                m.d.sync += self.rsp_rd.eq(0)    # Return 0 (success)

            # Function ID 2: Push Activations and Calculate
            with m.Elif(self.cmd_func_id == 2):
                m.d.comb += [
                    array.row0_in.eq(self.cmd_rs1[0:8]),
                    array.row1_in.eq(self.cmd_rs2[0:8]),
                    array.en.eq(1) # Enable array pulse
                ]
                # Wait for array to pulse, then return the top-left accumulator as an example
                m.d.sync += self.rsp_valid.eq(1)
                m.d.sync += self.rsp_rd.eq(array.out00)

        return m

In [ ]:
#include "cfu.h"

// Custom macros provided by the LiteX/CFU toolchain
#define FIKRA_LOAD_WEIGHTS(weights) cfu_op0(1, weights, 0)
#define FIKRA_PUSH_ACTS(row0, row1) cfu_op0(2, row0, row1)

void run_fikra_layer() {
    uint32_t packed_weights = 0x01FF01FF; // Packed ternary weights
    uint32_t act_row0 = 0x0000000A;       // Activation value 10
    uint32_t act_row1 = 0x00000014;       // Activation value 20

    // 1. Send instruction to NPU to load weights
    FIKRA_LOAD_WEIGHTS(packed_weights);

    // 2. Push activations into the NPU and instantly get the result
    uint32_t hardware_result = FIKRA_PUSH_ACTS(act_row0, act_row1);

    printf("Hardware Accelerator Output: %d\n", hardware_result);
}

SyntaxError: invalid syntax (3112288838.py, line 3)

In [9]:
from amaranth import *
from amaranth.sim import Simulator, Tick

# 1. The CFU Wrapper (The blueprint from the previous step)
class FikraCFU(Elaboratable):
    def __init__(self):
        self.cmd_valid = Signal()
        self.cmd_func_id = Signal(10)
        self.cmd_rs1 = Signal(32)
        self.cmd_rs2 = Signal(32)

        self.rsp_valid = Signal()
        self.rsp_rd = Signal(32)

    def elaborate(self, platform):
        m = Module()
        # Instantiate the array we built earlier
        m.submodules.array = array = SystolicArray2x2()

        # Default state
        m.d.sync += self.rsp_valid.eq(0)

        with m.If(self.cmd_valid):
            # ID 1: Load Weights
            with m.If(self.cmd_func_id == 1):
                m.d.comb += [
                    array.w00.eq(self.cmd_rs1[0:8]),
                    array.w01.eq(self.cmd_rs1[8:16]),
                    array.w10.eq(self.cmd_rs1[16:24]),
                    array.w11.eq(self.cmd_rs1[24:32]),
                ]
                m.d.sync += self.rsp_valid.eq(1)
                m.d.sync += self.rsp_rd.eq(0)

            # ID 2: Push Activations
            with m.Elif(self.cmd_func_id == 2):
                m.d.comb += [
                    array.row0_in.eq(self.cmd_rs1[0:8]),
                    array.row1_in.eq(self.cmd_rs2[0:8]),
                    array.en.eq(1)
                ]
                m.d.sync += self.rsp_valid.eq(1)
                m.d.sync += self.rsp_rd.eq(array.out00)

        return m

# ==========================================
# 2. THE TESTBENCH (Faking the RISC-V CPU)
# ==========================================

dut = FikraCFU()
sim = Simulator(dut)
sim.add_clock(1e-6)

def cpu_process():
    print("⚡ Booting Virtual RISC-V CPU...")
    yield dut.cmd_valid.eq(0)
    yield Tick()

    # STEP 1: CPU Sends Weights
    print("💻 CPU: Sending FIKRA_LOAD_WEIGHTS instruction...")
    yield dut.cmd_valid.eq(1)
    yield dut.cmd_func_id.eq(1)

    # We pack 4 weights into a 32-bit register (rs1).
    # Hex 0x01_00_FF_01 translates to w11=1, w10=0, w01=-1 (FF in 8-bit), w00=1
    yield dut.cmd_rs1.eq(0x0100FF01)
    yield Tick()

    # Drop the valid signal and wait for NPU to respond
    yield dut.cmd_valid.eq(0)
    while not (yield dut.rsp_valid):
        yield Tick()
    print("✅ NPU: Weights successfully loaded into silicon.")

    # STEP 2: CPU Sends Activations
    print("💻 CPU: Sending FIKRA_PUSH_ACTS instruction (Row0=10, Row1=20)...")
    yield dut.cmd_valid.eq(1)
    yield dut.cmd_func_id.eq(2)
    yield dut.cmd_rs1.eq(10) # Data for row 0
    yield dut.cmd_rs2.eq(20) # Data for row 1
    yield Tick()

    yield dut.cmd_valid.eq(0)
    while not (yield dut.rsp_valid):
        yield Tick()

    # Read the final computation back from the NPU
    hardware_result = yield dut.rsp_rd
    print(f"🏁 NPU: Computation complete. Output Register = {hardware_result}")

sim.add_testbench(cpu_process)

with sim.write_vcd("lacesse_cfu_integration.vcd"):
    sim.run()

⚡ Booting Virtual RISC-V CPU...
💻 CPU: Sending FIKRA_LOAD_WEIGHTS instruction...
✅ NPU: Weights successfully loaded into silicon.
💻 CPU: Sending FIKRA_PUSH_ACTS instruction (Row0=10, Row1=20)...
🏁 NPU: Computation complete. Output Register = 0


In [10]:
from amaranth.back import verilog

# 1. Instantiate the top-level chip design
top_chip = FikraCFU()

# 2. We must explicitly tell the compiler which pins need to be physically wired to the outside world
external_pins = [
    top_chip.cmd_valid, top_chip.cmd_func_id,
    top_chip.cmd_rs1, top_chip.cmd_rs2,
    top_chip.rsp_valid, top_chip.rsp_rd
]

# 3. Compile the Python into raw Verilog and save it to a file
with open("lacesse_npu_core.v", "w") as f:
    f.write(verilog.convert(top_chip, ports=external_pins))

print("✅ Physical Blueprint Generated: lacesse_npu_core.v")

YosysError: Could not find an acceptable Yosys binary. The `amaranth-yosys` PyPI package, if available for this platform, can be used as fallback

In [12]:
!pip install amaranth-yosys

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 100.2 MB/s eta 0:00:00


In [13]:
from amaranth.back import verilog

# 1. Instantiate the top-level chip design
top_chip = FikraCFU()

# 2. We must explicitly tell the compiler which pins need to be physically wired to the outside world
external_pins = [
    top_chip.cmd_valid, top_chip.cmd_func_id,
    top_chip.cmd_rs1, top_chip.cmd_rs2,
    top_chip.rsp_valid, top_chip.rsp_rd
]

# 3. Compile the Python into raw Verilog and save it to a file
with open("lacesse_npu_core.v", "w") as f:
    f.write(verilog.convert(top_chip, ports=external_pins))

print("✅ Physical Blueprint Generated: lacesse_npu_core.v")

✅ Physical Blueprint Generated: lacesse_npu_core.v


In [14]:
from amaranth import *
from amaranth.back import verilog

# We assume FikraCFU is still in memory from the previous cell

class TTWrapper(Elaboratable):
    def __init__(self):
        # 1. Standard Tiny Tapeout Physical Pins
        self.ui_in   = Signal(8)  # Dedicated inputs from the physical switches
        self.uo_out  = Signal(8)  # Dedicated outputs to the physical LEDs
        self.uio_in  = Signal(8)  # Bidirectional (Inputs)
        self.uio_out = Signal(8)  # Bidirectional (Outputs)
        self.uio_oe  = Signal(8)  # Output Enable
        self.ena     = Signal()   # Always 1 when active
        self.clk     = Signal()   # Physical clock from the PCB
        self.rst_n   = Signal()   # Active-low reset button

    def elaborate(self, platform):
        m = Module()

        # Instantiate the Fikra NPU we built
        m.submodules.npu = npu = FikraCFU()

        # 2. The Serialization Register
        # We need a 32-bit holding tank because we can only load 8 bits at a time from ui_in
        shift_reg = Signal(32)

        # Pin Assignment:
        # ui_in[0] = Shift enable (1 to pull data in)
        # ui_in[1] = NPU Command Valid
        # ui_in[2:4] = NPU Function ID
        # ui_in[4:8] = 4 bits of raw data coming in

        with m.If(self.ui_in[0]):
            # Shift the existing data to the left, and append the 4 new incoming bits
            m.d.sync += shift_reg.eq(Cat(self.ui_in[4:8], shift_reg[0:28]))

        # 3. Wiring the wrapper to the NPU core
        m.d.comb += [
            npu.cmd_valid.eq(self.ui_in[1]),
            npu.cmd_func_id.eq(self.ui_in[2:4]),
            npu.cmd_rs1.eq(shift_reg), # Feed the fully assembled 32-bit tank into the NPU

            # Output the bottom 8 bits of the NPU's result directly to the physical LEDs
            self.uo_out.eq(npu.rsp_rd[0:8]),

            # TinyTapeout requires us to declare if bidirectional pins are acting as in or out.
            # We set all to output (0xFF) and drive them to 0 to keep the chip clean.
            self.uio_oe.eq(0xFF),
            self.uio_out.eq(0x00)
        ]

        return m

# Instantiate the wrapper
top = TTWrapper()

# Tell the compiler exactly which pins touch the physical exterior of the chip
tt_pins = [top.ui_in, top.uo_out, top.uio_in, top.uio_out, top.uio_oe, top.ena, top.clk, top.rst_n]

# Export the final tapeout Verilog file
with open("tt_um_lacesse_npu.v", "w") as f:
    f.write(verilog.convert(top, ports=tt_pins, name="tt_um_lacesse_npu"))

print("✅ Fab-Ready Wrapper Generated: tt_um_lacesse_npu.v")

✅ Fab-Ready Wrapper Generated: tt_um_lacesse_npu.v


In [15]:
# Update the package manager and install Icarus Verilog
!apt-get update -qq
!apt-get install -y iverilog

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  gtkwave
The following NEW packages will be installed:
  iverilog
0 upgraded, 1 newly installed, 0 to remove and 11 not upgraded.
Need to get 2,130 kB of archives.
After this operation, 6,749 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 iverilog amd64 11.0-1.1 [2,130 kB]
Fetched 2,130 kB in 1s (2,850 kB/s)
Selecting previously unselected package iverilog.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../iverilog_11.0-1.1_amd64.deb ...
Unpacking iverilog (11.0-1.1) ...
Setting up iverilog (11.0-1.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [16]:
# Feed your final wrapper file into the compiler
!iverilog -o lacesse_fab_test.vvp tt_um_lacesse_npu.v

# If the command above succeeds without printing any syntax errors,
# it means your hardware is mathematically flawless.
!echo "✅ Compilation Successful: The Fab computers can read your silicon design."

✅ Compilation Successful: The Fab computers can read your silicon design.
